In [1]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv(override=True)

region = os.environ.get("BEDROCK_REGION", "us-west-2")
client = boto3.client("bedrock-runtime", region_name=region)
model_id = os.environ["BEDROCK_MODEL_ID"]

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

In [3]:
messages = []

add_user_message(messages, "Is coffee or  tea better for breakfast?")
add_assistant_message(messages, "They are the same because")

chat(messages)

' both tea and coffee contain caffeine. Tea and coffee are different types of drinks. Tea is derived from the Camellia sinensis plant, while coffee is derived from coffee beans. However, what makes both drinks popular is their caffeine content, which provides energy that is very beneficial for those who start their day.\n\nHowever, there are several differences:\n- **Coffee** generally has higher caffeine content (95mg per cup vs 25-50mg for tea), making it more effective for a quick energy boost\n- **Tea** contains L-theanine, which can promote calm alertness and may reduce caffeine jitters\n- **Coffee** may be better for metabolism and contains antioxidants\n- **Tea** (especially green tea) has its own set of beneficial antioxidants and may be gentler on the stomach\n\nThe "better" choice depends on your personal preferences, caffeine tolerance, and how your body responds to each. Some people prefer coffee\'s stronger kick, while others like tea\'s gentler, more sustained energy. Bot

In [4]:
messages = []

add_user_message(messages, "Count from 1 to 10")

chat(messages, stop_sequences=["5", "3, 4"])

'1, 2, '

In [5]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

##### Exercise!

- Use message prefilling and stop sequences _only_ to get three different commands in a single response
- There shouldn't be any comments or explanation
- Hint: message prefilling isn't limited to just characters like ```


In [6]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(
    messages,
    "Here are all three commands in a single block without any comments:\n```bash",
)

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws lambda list-functions'